# Анализ лог-данных производственных сенсоров

In [1]:
!pip install pyspark py4j

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("reduceByKey_Example") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

## Загрузка и предварительная обработка

### Создание rdd

In [9]:
raw_data = """2025-01-10 08:00:05|S101|25.3|10.1|OK|NULL
2025-01-10 08:05:10|S102|26.1|NULL|WARNING|E01
2025-01-10 08:10:00|S101|25.5|10.2|OK|NULL
2025-01-10 08:15:30|S103|NULL|9.9|ERROR|E05
2025-01-10 08:20:15|S102|25.8|10.0|OK|NULL
2025-01-10 08:25:00|S101|25.4|10.1|OK|NULL
2025-01-10 08:30:20|S104|27.0|10.3|OK|NULL
2025-01-10 08:35:05|S101|25.6|NULL|OK|NULL
2025-01-10 08:40:00|S103|24.9|9.8|WARNING|NULL
2025-01-10 08:45:10|S102|26.0|10.1|OK|NULL
2025-01-11 09:00:00|S104|27.2|10.4|ERROR|E03
2025-01-11 09:05:30|S101|25.7|10.2|OK|NULL
2025-01-11 09:10:00|S105|24.0|9.5|OK|NULL
2025-01-11 09:15:10|S101|25.8|10.3|OK|NULL
2025-01-11 09:20:20|S102|26.2|NULL|WARNING|E02
2025-01-11 09:25:00|S103|25.0|9.7|OK|NULL
2025-01-12 10:00:15|S104|NULL|10.5|ERROR|E04
2025-01-12 10:05:00|S105|24.1|9.6|OK|NULL
2025-01-12 10:10:05|S101|25.9|10.4|OK|NULL
2025-01-12 10:15:00|S102|26.3|10.2|OK|NULL
2025-01-12 10:20:10|S103|NULL|NULL|ERROR|E05
2025-01-13 11:00:00|S104|27.3|10.6|OK|NULL
2025-01-13 11:05:15|S105|24.2|9.7|WARNING|E01
2025-01-13 11:10:00|S101|26.0|10.5|OK|NULL
2025-01-13 11:15:20|S102|NULL|10.3|OK|NULL
2025-01-13 11:20:00|S103|25.1|9.9|OK|NULL
2025-01-14 08:30:10|S104|27.4|NULL|ERROR|E03
2025-01-14 08:35:00|S105|24.3|9.8|OK|NULL
2025-01-14 08:40:15|S101|26.1|10.6|OK|NULL
2025-01-14 08:45:00|S102|26.4|10.4|OK|NULL
2025-01-14 08:50:10|S103|25.2|10.0|WARNING|NULL
2025-01-15 09:00:00|S104|27.5|10.7|OK|NULL
2025-01-15 09:05:15|S105|NULL|9.9|ERROR|E02
2025-01-15 09:10:00|S101|26.2|10.7|OK|NULL
2025-01-15 09:15:20|S102|26.5|NULL|OK|NULL
2025-01-15 09:20:00|S103|25.3|10.1|OK|NULL
2025-01-16 10:00:10|S104|27.6|10.8|OK|NULL
2025-01-16 10:05:00|S105|24.5|10.0|WARNING|E04
2025-01-16 10:10:15|S101|26.3|10.8|OK|NULL
2025-01-16 10:15:00|S102|26.6|10.5|OK|NULL"""


In [10]:
rdd_logs = sc.parallelize(raw_data.splitlines())
rdd_logs.collect()


['2025-01-10 08:00:05|S101|25.3|10.1|OK|NULL',
 '2025-01-10 08:05:10|S102|26.1|NULL|WARNING|E01',
 '2025-01-10 08:10:00|S101|25.5|10.2|OK|NULL',
 '2025-01-10 08:15:30|S103|NULL|9.9|ERROR|E05',
 '2025-01-10 08:20:15|S102|25.8|10.0|OK|NULL',
 '2025-01-10 08:25:00|S101|25.4|10.1|OK|NULL',
 '2025-01-10 08:30:20|S104|27.0|10.3|OK|NULL',
 '2025-01-10 08:35:05|S101|25.6|NULL|OK|NULL',
 '2025-01-10 08:40:00|S103|24.9|9.8|WARNING|NULL',
 '2025-01-10 08:45:10|S102|26.0|10.1|OK|NULL',
 '2025-01-11 09:00:00|S104|27.2|10.4|ERROR|E03',
 '2025-01-11 09:05:30|S101|25.7|10.2|OK|NULL',
 '2025-01-11 09:10:00|S105|24.0|9.5|OK|NULL',
 '2025-01-11 09:15:10|S101|25.8|10.3|OK|NULL',
 '2025-01-11 09:20:20|S102|26.2|NULL|WARNING|E02',
 '2025-01-11 09:25:00|S103|25.0|9.7|OK|NULL',
 '2025-01-12 10:00:15|S104|NULL|10.5|ERROR|E04',
 '2025-01-12 10:05:00|S105|24.1|9.6|OK|NULL',
 '2025-01-12 10:10:05|S101|25.9|10.4|OK|NULL',
 '2025-01-12 10:15:00|S102|26.3|10.2|OK|NULL',
 '2025-01-12 10:20:10|S103|NULL|NULL|ERROR|E05

In [11]:
field_logs = rdd_logs.map(lambda x: x.split("|"))
field_logs.collect()

[['2025-01-10 08:00:05', 'S101', '25.3', '10.1', 'OK', 'NULL'],
 ['2025-01-10 08:05:10', 'S102', '26.1', 'NULL', 'WARNING', 'E01'],
 ['2025-01-10 08:10:00', 'S101', '25.5', '10.2', 'OK', 'NULL'],
 ['2025-01-10 08:15:30', 'S103', 'NULL', '9.9', 'ERROR', 'E05'],
 ['2025-01-10 08:20:15', 'S102', '25.8', '10.0', 'OK', 'NULL'],
 ['2025-01-10 08:25:00', 'S101', '25.4', '10.1', 'OK', 'NULL'],
 ['2025-01-10 08:30:20', 'S104', '27.0', '10.3', 'OK', 'NULL'],
 ['2025-01-10 08:35:05', 'S101', '25.6', 'NULL', 'OK', 'NULL'],
 ['2025-01-10 08:40:00', 'S103', '24.9', '9.8', 'WARNING', 'NULL'],
 ['2025-01-10 08:45:10', 'S102', '26.0', '10.1', 'OK', 'NULL'],
 ['2025-01-11 09:00:00', 'S104', '27.2', '10.4', 'ERROR', 'E03'],
 ['2025-01-11 09:05:30', 'S101', '25.7', '10.2', 'OK', 'NULL'],
 ['2025-01-11 09:10:00', 'S105', '24.0', '9.5', 'OK', 'NULL'],
 ['2025-01-11 09:15:10', 'S101', '25.8', '10.3', 'OK', 'NULL'],
 ['2025-01-11 09:20:20', 'S102', '26.2', 'NULL', 'WARNING', 'E02'],
 ['2025-01-11 09:25:00', '

Приводям поля к нужным типам

In [12]:
parsed_rdd = field_logs.map(lambda f: (
    f[0],                     # timestamp
    f[1],                     # sensor_id
    float(f[2]) if f[2] != "NULL" else None,  # temperature
    float(f[3]) if f[3] != "NULL" else None,  # humidity
    f[4],                     # status
    f[5] if f[5] != "NULL" else None           # error_code
))

## AD-hoc

Подсчет общего количества записей по статусам

In [13]:
status_counts = (
    parsed_rdd
    .map(lambda x: (x[4], 1))
    .reduceByKey(lambda a, b: a + b)
)

print(status_counts.collect())

[('WARNING', 6), ('OK', 28), ('ERROR', 6)]


Подсчет активных сенсоров с ошибками

In [14]:
errors_sensor_id = field_logs.filter(lambda x: x[4].upper() == 'ERROR')
errors_sensor_id.collect()

[['2025-01-10 08:15:30', 'S103', 'NULL', '9.9', 'ERROR', 'E05'],
 ['2025-01-11 09:00:00', 'S104', '27.2', '10.4', 'ERROR', 'E03'],
 ['2025-01-12 10:00:15', 'S104', 'NULL', '10.5', 'ERROR', 'E04'],
 ['2025-01-12 10:20:10', 'S103', 'NULL', 'NULL', 'ERROR', 'E05'],
 ['2025-01-14 08:30:10', 'S104', '27.4', 'NULL', 'ERROR', 'E03'],
 ['2025-01-15 09:05:15', 'S105', 'NULL', '9.9', 'ERROR', 'E02']]

In [15]:
errors_sensor_id_onlu_sensor_id = errors_sensor_id.map(lambda x: (x[1], 1))
errors_sensor_id_onlu_sensor_id.reduceByKey(lambda a, b: a + b).collect()

[('S105', 1), ('S103', 2), ('S104', 3)]

Расчет среднего значения температуры:

In [16]:
avg_temperature = field_logs.filter(lambda x: x[2] != "NULL").map(lambda x: float(x[2])).mean()
round(avg_temperature, 2)

25.84

Общее количество ошибок по коду

In [17]:
count_errors = field_logs.filter(lambda x: x[5].upper() != 'NULL').map(lambda x: (x[5], 1))
count_errors.reduceByKey(lambda a, b: a + b).collect()

[('E01', 2), ('E02', 2), ('E05', 2), ('E03', 2), ('E04', 2)]

Высокое давление и температура

In [18]:
high_temperature_and_davlenia = field_logs.filter(lambda x: x[2] != 'NULL' and x[3] != 'NULL' and float(x[2]) > 26.0 and float(x[3]) > 10.0)
result = high_temperature_and_davlenia.map(lambda x: (x[0], x[1], x[2], x[3])).collect()
result

[('2025-01-10 08:30:20', 'S104', '27.0', '10.3'),
 ('2025-01-11 09:00:00', 'S104', '27.2', '10.4'),
 ('2025-01-12 10:15:00', 'S102', '26.3', '10.2'),
 ('2025-01-13 11:00:00', 'S104', '27.3', '10.6'),
 ('2025-01-14 08:40:15', 'S101', '26.1', '10.6'),
 ('2025-01-14 08:45:00', 'S102', '26.4', '10.4'),
 ('2025-01-15 09:00:00', 'S104', '27.5', '10.7'),
 ('2025-01-15 09:10:00', 'S101', '26.2', '10.7'),
 ('2025-01-16 10:00:10', 'S104', '27.6', '10.8'),
 ('2025-01-16 10:10:15', 'S101', '26.3', '10.8'),
 ('2025-01-16 10:15:00', 'S102', '26.6', '10.5')]

# Анализ списка распространенных паролей

### Загрузка и предварительная обработка

In [27]:
rdd_password = sc.textFile("/content/drive/MyDrive/py_spark_data/password.txt")
rdd_password = rdd_password.map(lambda x: x.strip())
rdd_password = rdd_password.filter(lambda x: len(x) != 0)

### AD-HOC

Средняя, максимальная и минимальная длина пароля

In [33]:
avg_len_password = rdd_password.map(lambda x: len(x))

print(f'Средняя длина пароля - {int(avg_len_password.mean())}')
print(f'Максимальная длина пароля - {int(avg_len_password.max())}')
print(f'Минимальная длина пароля - {int(avg_len_password.min())}')

Средняя длина пароля - 9
Максимальная длина пароля - 15
Минимальная длина пароля - 6


Топ 5 длин паролей

In [42]:
result = avg_len_password.countByValue()
top_5 = sorted(result.items(), key=lambda x: x[1], reverse=True)[:5]
top_5

[(6, 215887), (8, 122106), (7, 115450), (10, 113429), (9, 110343)]

Символьный состав

In [47]:
rdd_password_only_alpha = rdd_password.filter(lambda x: x.isalpha())
print(f'Количество паролей состоящие только из букв - {rdd_password_only_alpha.count()}')
rdd_password_only_number = rdd_password.filter(lambda x: x.isdigit())
print(f'Количество паролей состоящие только из цифр - {rdd_password_only_number.count()}')

Количество паролей состоящие только из букв - 115088
Количество паролей состоящие только из цифр - 441347


Распространенные префиксы/суффиксы:

In [54]:
rdd_password_prefix_3 = rdd_password.map(lambda x: x[:3])
result = rdd_password_prefix_3.countByValue()
top_5 = sorted(result.items(), key=lambda x: x[1], reverse=True)[:5]
top_5

[('123', 17857), ('qwe', 17054), ('pas', 13290), ('654', 9925), ('edc', 9813)]

Лучше так, потому что работает распределенно и не тащит на драйвер

In [55]:
rdd_password_suffixes_3 = rdd_password.map(lambda x: (x[-3:], 1))
rdd_password_suffixes_3 = rdd_password_suffixes_3.reduceByKey(lambda x, y: x + y)
top_suffixes = rdd_password_suffixes_3.top(5, key = lambda x: x[1])
top_suffixes

[('123', 16135),
 ('789', 15266),
 ('020', 14636),
 ('021', 14558),
 ('022', 14552)]

# Анализ словаря русских слов

### Обработка данных

In [60]:
rdd_russian_dict = sc.textFile("/content/drive/MyDrive/py_spark_data/Rus_dict.txt")
rdd_russian_dict = rdd_russian_dict.map(lambda x: x.strip())
rdd_russian_dict.top(3)

['ёршик', 'ёрш', 'ёрничество']

### AD-HOC

Статистика по словам:

In [71]:
print(f'Общее количество слова - {rdd_russian_dict.count()}')
rdd_len_word = rdd_russian_dict.map(lambda x: (x, len(x)))
min_len_word = rdd_len_word.takeOrdered(5, key = lambda x: x[1])
print(f'Самые короткие слова - {min_len_word}')
max_len_word = rdd_len_word.top(5, key = lambda x: x[1])
print(f'Самые длинные слова - {max_len_word}')

Общее количество слова - 51301
Самые короткие слова - [('ад', 2), ('аз', 2), ('аи', 2), ('ар', 2), ('ас', 2)]
Самые длинные слова - [('человеконенавистничество', 24), ('интернационализирование', 23), ('неусовершенствованность', 23), ('переосвидетельствование', 23), ('адмиралтейств-коллегия', 22)]


3.1. Посчитайте, сколько слов имеют каждую конкретную длину (например, слов длиной 3: X, слов длиной 5: Y). Отсортируйте результат по длине.

In [88]:
rdd_russian_dict_len_word = rdd_russian_dict.map(lambda x: (len(x), 1))
rdd_russian_dict_len_word_group = rdd_russian_dict_len_word.reduceByKey(lambda x, y: x + y)
rdd_russian_dict_len_word_group.sortByKey().collect()

[(2, 54),
 (3, 472),
 (4, 1607),
 (5, 3483),
 (6, 4836),
 (7, 6335),
 (8, 6885),
 (9, 6612),
 (10, 5501),
 (11, 4591),
 (12, 3527),
 (13, 2774),
 (14, 1994),
 (15, 1123),
 (16, 692),
 (17, 364),
 (18, 227),
 (19, 117),
 (20, 60),
 (21, 29),
 (22, 14),
 (23, 3),
 (24, 1)]

3.2. Определите длину слова, которая встречается чаще всего.

In [87]:
print(f'Самая частая длина - {rdd_russian_dict_len_word_group.max(key = lambda x: x[1])[0]}')

Самая частая длина - 8


4. Символьный анализ:

4.1. Посчитайте, сколько слов содержат букву 'ё' .

In [91]:
rdd_russian_dict_not_e = rdd_russian_dict.filter(lambda x: 'ё' in x)
rdd_russian_dict_not_e.count()

2111

4.2. Отфильтруйте и выведите все слова, которые являются палиндромами (читаются одинаково вперед и назад, например, "шалаш", "заказ").

In [92]:
rdd_russian_dict_polindrom = rdd_russian_dict.filter(lambda x: x[::-1] == x)
rdd_russian_dict_polindrom.collect()

['ага',
 'боб',
 'дед',
 'довод',
 'доход',
 'заказ',
 'кабак',
 'казак',
 'киник',
 'кок',
 'колок',
 'комок',
 'косок',
 'коток',
 'лал',
 'мадам',
 'мим',
 'наган',
 'нойон',
 'око',
 'поп',
 'потоп',
 'пуп',
 'радар',
 'репер',
 'ротатор',
 'ротор',
 'тат',
 'тет-а-тет',
 'топот',
 'тот',
 'тут',
 'ушу',
 'шалаш',
 'шиш']

# Анализ логов веб-сервера

### Загрузка данных

In [98]:
import re
rdd_logs_web_server = sc.textFile('/content/drive/MyDrive/py_spark_data/logfiles.log')
def parse_log(log_line):
    pattern = r'(\S+) - - \[(.*?)\] "(.*?)" (\d{3}) (\d+)'
    match = re.match(pattern, log_line)
    if match:
        ip = match.group(1)
        request = match.group(3)
        status_code = int(match.group(4))
        response_size = int(match.group(5))
        return (ip, request, status_code, response_size)
    return None

rdd_logs_web_server = rdd_logs_web_server.map(lambda x: parse_log(x))
rdd_logs_web_server.top(3)

[('99.94.73.106', 'DELETE /auth/login HTTP/1.0', 200, 5020),
 ('99.93.151.237', 'GET /search?q=spark HTTP/1.0', 401, 4993),
 ('99.92.127.145', 'HEAD /admin/dashboard HTTP/1.0', 302, 4914)]

### AD-HOC

2. Общая статистика по логам

2.1. Посчитайте общее количество запросов.

2.2.Рассчитайте средний размер ответа сервера (в байтах) по всем запросам.

2.3. Определите количество уникальных IP-адресов, которые обращались к серверу.

In [99]:
rdd_logs_web_server.count()

10000

In [101]:
rdd_logs_web_server_only_byte = rdd_logs_web_server.map(lambda x: x[3])
print(f'Средний размер ответа сервера в байтах - {int(rdd_logs_web_server_only_byte.mean())}')

Средний размер ответа сервера в байтах - 4999


In [104]:
rdd_logs_web_server.map(lambda x: x[0]).distinct().count()

10000

3. Анализ HTTP-статусов

3.1. Посчитайте количество запросов для каждого HTTP-статус-кода (например, сколько 200, сколько 404, сколько 500).

3.2. Определите долю успешных запросов (статус-код 200) от общего числа запросов в процента


In [107]:
result_status = rdd_logs_web_server.map(lambda x: (x[2], 1))
result_status.reduceByKey(lambda x, y: x + y).collect()

[(200, 3850),
 (500, 797),
 (302, 773),
 (404, 820),
 (502, 730),
 (400, 734),
 (401, 781),
 (301, 780),
 (403, 735)]

In [109]:
status_200 = rdd_logs_web_server.filter(lambda x: str(x[2]) == '200').count()
status_all = rdd_logs_web_server.count()
print(f'Доля успешных запросов {status_200 / status_all * 100} %')

Доля успешных запросов 38.5 %


4. Анализ эндпоинтов и запросов

4.1. Найдите Топ-5 самых часто запрашиваемых эндпоинтов. Эндпоинт - это часть URL-адреса, которая указывает на конкретный ресурс или функцию на сервере, к которой обращается клиент. Например, в запросе "GET /api/v1/users HTTP/1.0" эндпоинтом является /api/v1/users.

4.2. Посчитайте, сколько запросов каждого типа (GET, POST, PUT и т.д.) было сделано

In [118]:
end_point_test = rdd_logs_web_server.map(lambda x: ((x[1].split(' '))[1], 1))
top5_end_point_test = end_point_test.reduceByKey(lambda x, y: x + y)
top5_end_point_test.top(5)

[('/search?q=spark', 952),
 ('/images/logo.png', 883),
 ('/docs/api', 949),
 ('/blog/latest', 837),
 ('/auth/register', 929)]

In [120]:
get_point_test = rdd_logs_web_server.map(lambda x: ((x[1].split(' '))[0], 1))
top5_end_point_test = get_point_test.reduceByKey(lambda x, y: x + y)
top5_end_point_test.top(5)

[('PUT', 1640),
 ('POST', 1641),
 ('OPTIONS', 1662),
 ('HEAD', 1669),
 ('GET', 1677)]

# Анализ вовлеченности постов в социальной сети

1. Загрузка и предварительная обработка данных

1.1. Загрузите данные о постах в RDD. Предположим, что файл называется posts.csv.

1.2. Исключите из RDD строку с заголовками (названиями столбцов)

1.3. Напишите функцию парсинга , которая будет преобразовывать каждую строку в кортеж Python. При этом:
* Post_id преобразуйте в int.
* comments преобразуйте в int.
* likes преобразуйте в int.
* Post_Type оставьте как строку.

In [128]:
posts_rdd = sc.textFile("/content/drive/MyDrive/py_spark_data/posts.csv")
header = posts_rdd.first()
posts_no_header_rdd = posts_rdd.filter(lambda x: x != header)
def parse_post(line):
    fields = line.split(",")

    post_id = int(fields[0])
    post_type = fields[1]
    comments = int(fields[2])
    likes = int(fields[3])

    return (post_id, post_type, comments, likes)
parsed_posts_rdd = posts_no_header_rdd.map(parse_post)
parsed_posts_rdd.take(5)

[(101, 'Carousel', 268, 16382),
 (102, 'Reel', 138, 9267),
 (103, 'Reel', 1089, 10100),
 (104, 'Reel', 271, 6943),
 (105, 'Reel', 145, 17158)]

2. Общая статистика вовлеченности

2.1. Посчитайте общее количество постов в датасете.

2.2. Рассчитайте общее количество лайков по всем постам.

2.3. Рассчитайте общее количество комментариев по всем постам.

2.4. Определите среднее количество лайков на один пост. Округлите результат до одного знака после запятой.

2.5. Определите среднее количество комментариев на один пост. Округлите результат до одного знака после запятой.

In [135]:
parsed_posts_rdd.map(lambda x: x[0]).count()

39

In [136]:
parsed_posts_rdd.map(lambda x: x[3]).sum()

707567

In [137]:
parsed_posts_rdd.map(lambda x: x[2]).sum()

8387

In [139]:
round(parsed_posts_rdd.map(lambda x: x[3]).sum() / parsed_posts_rdd.map(lambda x: x[0]).count(),1)

18142.7

In [140]:
round(parsed_posts_rdd.map(lambda x: x[2]).sum() / parsed_posts_rdd.map(lambda x: x[0]).count(),1)

215.1

3. Анализ по типам постов

3.1. Посчитайте количество постов каждого типа.

3.2. Для каждого типа поста рассчитайте среднее количество лайков.  Округлите результат до одного знака после запятой.

3.3. Для каждого типа поста рассчитайте среднее количество комментариев.  Округлите результат до одного знака после запятой.

In [143]:
count_type_posts = parsed_posts_rdd.map(lambda x: (x[1], 1))
count_type_posts = count_type_posts.reduceByKey(lambda x, y: x + y)
count_type_posts.collect()

[('Carousel', 14), ('Reel', 14), ('Image', 11)]

In [153]:
likes_carousel = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Carousel')\
                  .map(lambda x: x[3])\
                  .mean()
print(f'Среднее количество лайков с типом Carousel - {round(likes_carousel, 1)}')

comm_carousel = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Carousel')\
                  .map(lambda x: x[2])\
                  .mean()
print(f'Среднее количество комментариев с типом Carousel - {round(comm_carousel, 1)}')

Среднее количество лайков с типом Carousel - 20814.3
Среднее количество комментариев с типом Carousel - 174.3


In [155]:
likes_Reel = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Reel')\
                  .map(lambda x: x[3])\
                  .mean()
print(f'Среднее количество лайков с типом Reel - {round(likes_Reel, 1)}')

Comm_Reel = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Reel')\
                  .map(lambda x: x[2])\
                  .mean()
print(f'Среднее количество комментариев с типом Reel - {round(Comm_Reel, 1)}')



Среднее количество лайков с типом Reel - 15156.6
Среднее количество комментариев с типом Reel - 316.7


In [156]:
likes_Image = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Image')\
                  .map(lambda x: x[3])\
                  .mean()
round(likes_Image, 1)
print(f'Среднее количество лайков с типом Image - {round(likes_Reel, 1)}')

Image_Reel = parsed_posts_rdd\
                  .filter(lambda x: x[1] == 'Image')\
                  .map(lambda x: x[2])\
                  .mean()
print(f'Среднее количество комментариев с типом Image - {round(Image_Reel, 1)}')


Среднее количество лайков с типом Image - 15156.6
Среднее количество комментариев с типом Image - 137.5


4. Вовлеченность

4.1. Найдите Топ-5 постов с наибольшим количеством лайков. Выведите их Post_id, Post_Type , likes.

4.2. Найдите Топ-5 постов с наибольшим количеством комментариев. Выведите их Post_id, Post_Type, comments.

In [192]:
top_5_posts_like = parsed_posts_rdd.top(5, key=lambda x: x[3])

for post in top_5_posts_like:
    print(f"Post ID: {post[0]}, Type: {post[1]}, Likes: {post[3]}")

Post ID: 125, Type: Carousel, Likes: 79000
Post ID: 131, Type: Carousel, Likes: 59716
Post ID: 133, Type: Carousel, Likes: 58485
Post ID: 132, Type: Image, Likes: 53254
Post ID: 122, Type: Image, Likes: 50523


In [191]:
top_5_posts_comm = parsed_posts_rdd.top(5, key = lambda x: x[2])

for i in range(len(top_5_posts_comm)):
  print(top_5_posts_comm[i][0:3])

(103, 'Reel', 1089)
(109, 'Reel', 884)
(123, 'Reel', 555)
(131, 'Carousel', 507)
(125, 'Carousel', 466)


# Анализ стран

1. Загрузите файл List of Countries.txt в RDD.

In [194]:
rdd_countries = sc.textFile("/content/drive/MyDrive/py_spark_data/List of Countries.txt")

2. Определите количество стран для каждой первой буквы их названия (сколько стран начинается на 'A', сколько на 'B' и так далее). Выведите результат в алфавитном порядке по букве.

In [216]:
rdd_countries_first_alpha = rdd_countries\
                              .map(lambda x: (x[0], 1))\
                              .reduceByKey(lambda x, y: x + y)
                              .sortByKey()
rdd_countries_first_alpha.collect()

3. Найдите 5 самых длинных названий стран в списке. Если есть несколько стран с одинаковой максимальной длиной, выведите их все.

In [219]:
country_len_rdd = rdd_countries.map(lambda x: (x, len(x)))

top_5_longest = (
    rdd_countries
    .map(lambda x: (x, len(x)))
    .takeOrdered(5, key=lambda x: -x[1])
)

threshold_len = top_5_longest[-1][1]

result = country_len_rdd.filter(lambda x: x[1] >= threshold_len)

result.collect()

[('Bosnia and Herzegovina', 22),
 ('Central African Republic', 24),
 ('Saint Kitts and Nevis', 21),
 ('Saint Vincent and the Grenadines', 32),
 ('United Arab Emirates', 20)]

4. Найдите 5 самых коротких названий стран в списке. Аналогично, если есть несколько с одинаковой минимальной длиной, выведите их все.

In [222]:
country_len_rdd = rdd_countries.map(lambda x: (x, len(x)))

top_5_short = (
    rdd_countries
    .map(lambda x: (x, len(x)))
    .takeOrdered(5, key=lambda x: x[1])
)

threshold_len = top_5_short[-1][1]

result = country_len_rdd.filter(lambda x: x[1] <= threshold_len)

result.collect()

[('Chad', 4),
 ('Cuba', 4),
 ('Iran', 4),
 ('Iraq', 4),
 ('Laos', 4),
 ('Mali', 4),
 ('Oman', 4),
 ('Peru', 4),
 ('Togo', 4)]

5. Рассчитайте среднюю длину названия стран в списке. Округлите результат до одного знаков после запятой.

In [224]:
avg_len = country_len_rdd.map(lambda x: x[1]).mean()
round(avg_len, 1)

8.5

6. Найдите все страны, названия которых состоят из двух и более слов.

In [230]:
all_countries_2_word = rdd_countries.filter(lambda x: (len(x.split(' '))) >= 2)
all_countries_2_word.collect()

['Antigua and Barbuda',
 'Bosnia and Herzegovina',
 'Burkina Faso',
 'Cabo Verde',
 'Central African Republic',
 'Channel Islands',
 'Costa Rica',
 "Côte d'Ivoire",
 'Czech Republic',
 'Dominican Republic',
 'DR Congo',
 'El Salvador',
 'Equatorial Guinea',
 'Faeroe Islands',
 'French Guiana',
 'Holy See',
 'Hong Kong',
 'Isle of Man',
 'North Korea',
 'North Macedonia',
 'Saint Helena',
 'Saint Kitts and Nevis',
 'Saint Lucia',
 'Saint Vincent and the Grenadines',
 'San Marino',
 'Sao Tome & Principe',
 'Saudi Arabia',
 'Sierra Leone',
 'South Africa',
 'South Korea',
 'South Sudan',
 'Sri Lanka',
 'State of Palestine',
 'The Bahamas',
 'Trinidad and Tobago',
 'United Arab Emirates',
 'United Kingdom',
 'United States',
 'Western Sahara']